# Tutorial para acceder a la API de Spotify



## 1. ¿Qué es una API?

Una API es una forma de comunicarnos con una aplicación o servicio desde nuestro código.

En este caso, usaremos la API de Spotify para consultar información pública sobre canciones, artistas, álbumes y popularidad.

Importante: este ejemplo usa el método `client_credentials`, que sirve para consultar datos públicos del catálogo de Spotify. No sirve para acceder a playlists privadas ni datos personales de usuarios.

## 2. Crear una app en Spotify Developer


1. Entra a: https://developer.spotify.com/
2. Inicia sesión con tu cuenta de Spotify.
3. Entra al Dashboard.
4. Crea una nueva aplicación.
5. Copia tu `Client ID` y tu `Client Secret`.

El `Client ID` identifica tu aplicación.

El `Client Secret` funciona como una contraseña, NO debes compartirlo ni subirlo a GitHub.

## 3. Instalaciòn de bibliotecas necesarias
Usaremos:

- `requests`: para conectarnos a internet y consultar la API.
- `pandas`: para organizar los resultados en tablas.

In [1]:
# Instalaciòn de bibliotecas necesarias
# Si ya las tienes instaladas, puedes saltar esta celda.

!pip install requests pandas

## 4. Importaciòn de bibliotecas

In [2]:
# Bibliotece para hacer solicitudes HTTP a la API
import requests

# Bibliotece para codificar las credenciales en Base64
import base64
import pandas as pd

c:\Users\Jessan\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


## 5. Guardar tus credenciales

Copia aquí tu `Client ID` y tu `Client Secret`.

Importante: no compartas este notebook si contiene tus credenciales reales.

In [ ]:
# Coloca aquí tus credenciales de Spotify Developer
# Reemplaza los textos por tus datos reales

ID_CLIENTE = "e3eca03511004e08b9ac6ec7e48597b9"
SECRETO_CLIENTE = "coloca_tu_client_secret"

## 6. Obtener el token de acceso

Para consultar la API necesitamos un token de acceso.

El token es un permiso temporal que Spotify nos da para hacer consultas.

In [4]:
def obtener_token_acceso(id_cliente, secreto_cliente):
    """
    Obtiene un token de acceso para consultar la API de Spotify.
    
    Parámetros:
    id_cliente: Client ID de la app de Spotify.
    secreto_cliente: Client Secret de la app de Spotify.
    
    Retorna:
    Token de acceso en formato texto.
    """

    # Unimos el ID del cliente y el secreto del cliente
    texto_autenticacion = f"{id_cliente}:{secreto_cliente}"

    # Convertimos el texto a bytes
    bytes_autenticacion = texto_autenticacion.encode("utf-8")

    # Codificamos las credenciales en Base64
    autenticacion_base64 = base64.b64encode(bytes_autenticacion).decode("utf-8")

    # URL oficial de Spotify para solicitar el token
    url = "https://accounts.spotify.com/api/token"

    # Encabezados requeridos por Spotify
    encabezados = {
        "Authorization": f"Basic {autenticacion_base64}",
        "Content-Type": "application/x-www-form-urlencoded"
    }

    # Indicamos que usaremos el flujo client_credentials
    datos = {
        "grant_type": "client_credentials"
    }

    # Enviamos la solicitud a Spotify
    respuesta = requests.post(url, headers=encabezados, data=datos)

    # Si ocurre un error, Python mostrará el detalle
    respuesta.raise_for_status()

    # Extraemos el token de la respuesta
    token = respuesta.json()["access_token"]

    return token

## 7. Probar la conexión

Si tus credenciales son correctas, esta celda debe mostrar el mensaje: `Conexión realizada correctamente`.

In [5]:
# Obtenemos el token de acceso
token = obtener_token_acceso(ID_CLIENTE, SECRETO_CLIENTE)

print("Conexión realizada correctamente")

Conexión realizada correctamente


## 8. Buscar canciones

Ahora crearemos una función para buscar canciones en Spotify.

La función recibirá una palabra o nombre de artista, por ejemplo:


- `Taylor Swift`

In [6]:
def buscar_canciones(consulta, token, limite=10, mercado=None, mostrar_atributos=False):
    """
    Busca canciones en Spotify y regresa los resultados en una tabla de pandas.
    
    Parámetros:
    consulta: Texto que queremos buscar, por ejemplo el nombre de un artista.
    token: Token de acceso obtenido previamente.
    limite: Número máximo de canciones que queremos obtener.
    mercado: País del mercado musical. Si se deja en None, no se envía este parámetro.
    mostrar_atributos: Si es True, muestra los atributos disponibles de la primera canción.
    
    Retorna:
    DataFrame con información de canciones.
    """

    # URL oficial del buscador de Spotify
    url = "https://api.spotify.com/v1/search"

    # Encabezado con el token de acceso
    encabezados = {
        "Authorization": f"Bearer {token}"
    }

    # Parámetros de búsqueda básicos
    parametros = {
        "q": consulta,
        "type": "track",
        "limit": limite
    }

    # Agregamos el mercado solo si el usuario lo especifica
    if mercado is not None:
        parametros["market"] = mercado

    # Enviamos la solicitud a Spotify
    respuesta = requests.get(url, headers=encabezados, params=parametros)

    # Si hay error, mostramos información útil para entender qué pasó
    if respuesta.status_code != 200:
        print("Código de error:", respuesta.status_code)
        print("Respuesta de Spotify:", respuesta.text)
        return pd.DataFrame()

    # Convertimos la respuesta a formato JSON
    datos = respuesta.json()

    # Obtenemos la lista de canciones
    canciones = datos["tracks"]["items"]

    # Validamos si Spotify regresó resultados
    if len(canciones) == 0:
        print("No se encontraron canciones para esa búsqueda.")
        return pd.DataFrame()

    # Mostramos los atributos disponibles de la primera canción si se solicita
    if mostrar_atributos:
        print(canciones[0].keys())

    # Lista donde guardaremos los resultados
    resultados = []

    # Recorremos cada canción encontrada
    for cancion in canciones:
        resultados.append({
            "nombre_cancion": cancion["name"],
            "artista": cancion["artists"][0]["name"] if cancion["artists"] else None,
            "album": cancion["album"]["name"],
            "fecha_lanzamiento": cancion["album"]["release_date"],
            "duracion_minutos": round(cancion["duration_ms"] / 60000, 2),
            "explicita": cancion["explicit"],
            "numero_disco": cancion["disc_number"],
            "numero_pista": cancion["track_number"],
            "disponible": cancion.get("is_playable"),
            "id_spotify": cancion["id"],
            "uri_spotify": cancion["uri"],
            "url_spotify": cancion["external_urls"]["spotify"]
        })

    # Convertimos la lista en una tabla de pandas
    tabla = pd.DataFrame(resultados)

    return tabla

## 9. Hacer una búsqueda de prueba

Buscaremos canciones relacionadas con `Bad Bunny`.

In [7]:
tabla_canciones = buscar_canciones("Bad Bunny", token, limite=10)

tabla_canciones

,nombre_cancion,artista,album,fecha_lanzamiento,duracion_minutos,explicita,numero_disco,numero_pista,disponible,id_spotify,uri_spotify,url_spotify
0,BAILE INoLVIDABLE,Bad Bunny,DeBÍ TiRAR MáS FOToS,2025-01-05,6.13,True,1,3,True,2lTm559tuIvatlT1u0JYG2,spotify:track:2lTm559tuIvatlT1u0JYG2,https://open.spotify.com/track/2lTm559tuIvatlT...
1,DtMF,Bad Bunny,DeBÍ TiRAR MáS FOToS,2025-01-05,3.95,True,1,16,True,3sK8wGT43QFpWrvNQsrQya,spotify:track:3sK8wGT43QFpWrvNQsrQya,https://open.spotify.com/track/3sK8wGT43QFpWrv...
2,Ojitos Lindos,Bad Bunny,Un Verano Sin Ti,2022-05-06,4.30,False,1,14,True,3k3NWokhRRkEPhCzPmV8TW,spotify:track:3k3NWokhRRkEPhCzPmV8TW,https://open.spotify.com/track/3k3NWokhRRkEPhC...
3,Tití Me Preguntó,Bad Bunny,Un Verano Sin Ti,2022-05-06,4.06,True,1,4,True,1IHWl5LamUGEuP4ozKQSXZ,spotify:track:1IHWl5LamUGEuP4ozKQSXZ,https://open.spotify.com/track/1IHWl5LamUGEuP4...
4,Si Veo a Tu Mamá,Bad Bunny,YHLQMDLG,2020-02-29,2.85,False,1,1,True,41wtwzCZkXwpnakmwJ239F,spotify:track:41wtwzCZkXwpnakmwJ239F,https://open.spotify.com/track/41wtwzCZkXwpnak...
5,Andrea,Bad Bunny,Un Verano Sin Ti,2022-05-06,5.66,True,1,19,True,44XjoNvtwevktFKjvVe1vH,spotify:track:44XjoNvtwevktFKjvVe1vH,https://open.spotify.com/track/44XjoNvtwevktFK...
6,Moscow Mule,Bad Bunny,Un Verano Sin Ti,2022-05-06,4.10,True,1,1,True,6Xom58OOXk2SoU711L2IXO,spotify:track:6Xom58OOXk2SoU711L2IXO,https://open.spotify.com/track/6Xom58OOXk2SoU7...
7,NUEVAYoL,Bad Bunny,DeBÍ TiRAR MáS FOToS,2025-01-05,3.06,False,1,1,True,5TFD2bmFKGhoCRbX61nXY5,spotify:track:5TFD2bmFKGhoCRbX61nXY5,https://open.spotify.com/track/5TFD2bmFKGhoCRb...
8,EoO,Bad Bunny,DeBÍ TiRAR MáS FOToS,2025-01-05,3.41,True,1,15,True,6J5kc12BW5HuP3d7C3vvx8,spotify:track:6J5kc12BW5HuP3d7C3vvx8,https://open.spotify.com/track/6J5kc12BW5HuP3d...
9,Efecto,Bad Bunny,Un Verano Sin Ti,2022-05-06,3.55,True,1,10,True,5Eax0qFko2dh7Rl2lYs3bx,spotify:track:5Eax0qFko2dh7Rl2lYs3bx,https://open.spotify.com/track/5Eax0qFko2dh7Rl...


## 10. Revisar información general de la tabla

Podemos revisar cuántas filas y columnas tiene la tabla.

In [8]:
# Ver el tamaño de la tabla
tabla_canciones.shape

(10, 12)

In [9]:
# Ver los nombres de las columnas
tabla_canciones.columns

Index(['nombre_cancion', 'artista', 'album', 'fecha_lanzamiento',
       'duracion_minutos', 'explicita', 'numero_disco', 'numero_pista',
       'disponible', 'id_spotify', 'uri_spotify', 'url_spotify'],
      dtype='str')

## 11. Ordenar canciones por duraciòn


In [10]:
# Ordenamos las canciones de mayor a menor popularidad
tabla_ordenada = tabla_canciones.sort_values(by="duracion_minutos", ascending=False)

tabla_ordenada

,nombre_cancion,artista,album,fecha_lanzamiento,duracion_minutos,explicita,numero_disco,numero_pista,disponible,id_spotify,uri_spotify,url_spotify
0,BAILE INoLVIDABLE,Bad Bunny,DeBÍ TiRAR MáS FOToS,2025-01-05,6.13,True,1,3,True,2lTm559tuIvatlT1u0JYG2,spotify:track:2lTm559tuIvatlT1u0JYG2,https://open.spotify.com/track/2lTm559tuIvatlT...
5,Andrea,Bad Bunny,Un Verano Sin Ti,2022-05-06,5.66,True,1,19,True,44XjoNvtwevktFKjvVe1vH,spotify:track:44XjoNvtwevktFKjvVe1vH,https://open.spotify.com/track/44XjoNvtwevktFK...
2,Ojitos Lindos,Bad Bunny,Un Verano Sin Ti,2022-05-06,4.30,False,1,14,True,3k3NWokhRRkEPhCzPmV8TW,spotify:track:3k3NWokhRRkEPhCzPmV8TW,https://open.spotify.com/track/3k3NWokhRRkEPhC...
6,Moscow Mule,Bad Bunny,Un Verano Sin Ti,2022-05-06,4.10,True,1,1,True,6Xom58OOXk2SoU711L2IXO,spotify:track:6Xom58OOXk2SoU711L2IXO,https://open.spotify.com/track/6Xom58OOXk2SoU7...
3,Tití Me Preguntó,Bad Bunny,Un Verano Sin Ti,2022-05-06,4.06,True,1,4,True,1IHWl5LamUGEuP4ozKQSXZ,spotify:track:1IHWl5LamUGEuP4ozKQSXZ,https://open.spotify.com/track/1IHWl5LamUGEuP4...
1,DtMF,Bad Bunny,DeBÍ TiRAR MáS FOToS,2025-01-05,3.95,True,1,16,True,3sK8wGT43QFpWrvNQsrQya,spotify:track:3sK8wGT43QFpWrvNQsrQya,https://open.spotify.com/track/3sK8wGT43QFpWrv...
9,Efecto,Bad Bunny,Un Verano Sin Ti,2022-05-06,3.55,True,1,10,True,5Eax0qFko2dh7Rl2lYs3bx,spotify:track:5Eax0qFko2dh7Rl2lYs3bx,https://open.spotify.com/track/5Eax0qFko2dh7Rl...
8,EoO,Bad Bunny,DeBÍ TiRAR MáS FOToS,2025-01-05,3.41,True,1,15,True,6J5kc12BW5HuP3d7C3vvx8,spotify:track:6J5kc12BW5HuP3d7C3vvx8,https://open.spotify.com/track/6J5kc12BW5HuP3d...
7,NUEVAYoL,Bad Bunny,DeBÍ TiRAR MáS FOToS,2025-01-05,3.06,False,1,1,True,5TFD2bmFKGhoCRbX61nXY5,spotify:track:5TFD2bmFKGhoCRbX61nXY5,https://open.spotify.com/track/5TFD2bmFKGhoCRb...
4,Si Veo a Tu Mamá,Bad Bunny,YHLQMDLG,2020-02-29,2.85,False,1,1,True,41wtwzCZkXwpnakmwJ239F,spotify:track:41wtwzCZkXwpnakmwJ239F,https://open.spotify.com/track/41wtwzCZkXwpnak...


## 12. Ejercicio: buscar otro artista

Ahora puedes cambiar el texto de búsqueda.

In [11]:
# Ejercicio 1: cambia el nombre del artista
tabla_ejercicio = buscar_canciones("Shakira", token, limite=10)

tabla_ejercicio

,nombre_cancion,artista,album,fecha_lanzamiento,duracion_minutos,explicita,numero_disco,numero_pista,disponible,id_spotify,uri_spotify,url_spotify
0,Dai Dai,Shakira,Dai Dai,2026-05-14,3.72,False,1,1,True,0kosUz0jePvjiz4ctmR6wL,spotify:track:0kosUz0jePvjiz4ctmR6wL,https://open.spotify.com/track/0kosUz0jePvjiz4...
1,Waka Waka (Esto es Africa) (feat. Freshlyground),Shakira,Waka Waka (Esto es Africa) (feat. Freshlyground),2010-04-29,3.38,False,1,1,True,0W8nDs4H2cqxxAgszNMYO3,spotify:track:0W8nDs4H2cqxxAgszNMYO3,https://open.spotify.com/track/0W8nDs4H2cqxxAg...
2,Antología,Shakira,Pies Descalzos,1995-10-06,4.24,False,1,2,True,0KAqMRUSZwzG3dZLdDA4eH,spotify:track:0KAqMRUSZwzG3dZLdDA4eH,https://open.spotify.com/track/0KAqMRUSZwzG3dZ...
3,Loba,Shakira,Loba,2009-09-02,3.15,False,1,1,True,30XvfCUPHSK2rzxmPlJGFf,spotify:track:30XvfCUPHSK2rzxmPlJGFf,https://open.spotify.com/track/30XvfCUPHSK2rzx...
4,"Shakira: Bzrp Music Sessions, Vol. 53/66",Bizarrap,"Shakira: Bzrp Music Sessions, Vol. 53/66",2023-01-11,3.58,False,1,1,True,4nrPB8O7Y7wsOCJdgXkthe,spotify:track:4nrPB8O7Y7wsOCJdgXkthe,https://open.spotify.com/track/4nrPB8O7Y7wsOCJ...
5,"Ciega, Sordomuda",Shakira,Donde Estan Los Ladrones,1998-09-29,4.46,False,1,1,True,7jxHeJLVpnP7S08JFF4GBi,spotify:track:7jxHeJLVpnP7S08JFF4GBi,https://open.spotify.com/track/7jxHeJLVpnP7S08...
6,Addicted to You,Shakira,Sale el Sol,2010-10-19,2.46,False,1,5,True,4zy1s9GnxWsNzZp1688euA,spotify:track:4zy1s9GnxWsNzZp1688euA,https://open.spotify.com/track/4zy1s9GnxWsNzZp...
7,Las de la Intuición,Shakira,Fijación Oral Volumen 1 (Expanded Edition),2005-06-03,3.67,False,1,8,True,7DnMd3BxVOedyOyR7K3F8L,spotify:track:7DnMd3BxVOedyOyR7K3F8L,https://open.spotify.com/track/7DnMd3BxVOedyOy...
8,Loca (feat. El Cata),Shakira,Sale el Sol,2010-10-19,3.06,False,1,2,True,42k1KeBehAd83lrGt1okiC,spotify:track:42k1KeBehAd83lrGt1okiC,https://open.spotify.com/track/42k1KeBehAd83lr...
9,TQG,KAROL G,MAÑANA SERÁ BONITO,2023-02-24,3.30,True,1,6,True,0DWdj2oZMBFSzRsi2Cvfzf,spotify:track:0DWdj2oZMBFSzRsi2Cvfzf,https://open.spotify.com/track/0DWdj2oZMBFSzRs...


In [14]:
# Ejercicio 2: guardar la tabla en CSV
tabla_ejercicio.to_csv("canciones_shakira.csv", index=False) # 0,1,2,3

print("Archivo canciones_shakira.csv guardado correctamente")

Archivo canciones_shakira.csv guardado correctamente
